# Flow Matching & Rectified Flow

Wiki reference for [Flow Matching & Rectified Flow](https://ml-viz-ruby.vercel.app/wiki/flow-matching). To keep your own copy, use **File -> Save a copy in Drive**.

**The idea in one sentence.** Learn a *velocity field* that transports noise to data along an ODE; train it on trivially simple straight-line targets (condition on one endpoint) and let averaging recover the complex marginal field.

We train a small velocity network with **conditional flow matching** on 2-D two-moons data, sample by integrating the ODE, watch quality improve with step count, then **reflow** to straighten the trajectories so a couple of steps suffice.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.neural_network import MLPRegressor

plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#1a1d27', 'figure.facecolor': '#0f1117',
    'axes.edgecolor': '#3a3d4a', 'grid.color': '#2a2d3a',
    'text.color': '#e2e8f0', 'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})
rng = np.random.default_rng(0)

# Target data: two moons, standardized.
data, _ = make_moons(n_samples=4000, noise=0.06, random_state=0)
data = (data - data.mean(0)) / data.std(0)
print('data shape:', data.shape)

## 1 - The conditional (Dirac) trick

The marginal velocity field is intractable, so we condition on a single endpoint. For a noise sample `z` and a data point `d`, the path is the straight line `x_t = (1-t) z + t d` and its velocity is the constant `d - z`. These per-pair targets are trivial regressions; their expectation is the true marginal field.

In [ ]:
def make_pairs(data, n, rng):
    z = rng.standard_normal((n, 2))                 # noise endpoints
    d = data[rng.integers(0, len(data), size=n)]    # data endpoints (random coupling)
    return z, d

def cfm_batch(z, d, rng):
    t = rng.random((len(z), 1))                      # progress in [0,1]
    x_t = (1 - t) * z + t * d                        # point on the straight path
    v = d - z                                        # constant conditional velocity
    feat = np.hstack([x_t, t])                       # network input: (x, t)
    return feat, v

z, d = make_pairs(data, 40000, rng)
X, V = cfm_batch(z, d, rng)
print('input (x_t, t):', X.shape, ' target velocity:', V.shape)

## 2 - Train the velocity network and sample by integrating the ODE

In [ ]:
def train_field(X, V):
    net = MLPRegressor(hidden_layer_sizes=(128, 128), activation='tanh',
                       max_iter=60, random_state=0)
    net.fit(X, V)
    return net

def sample(net, z0, steps):
    x = z0.copy()
    for k in range(steps):
        t = np.full((len(x), 1), k / steps)
        v = net.predict(np.hstack([x, t]))
        x = x + v / steps                            # explicit Euler step
    return x

net = train_field(X, V)

def nn_dist(samples, data):
    # mean distance from each sample to its nearest true data point (lower = better)
    d2 = ((samples[:, None, :] - data[None, :, :]) ** 2).sum(-1)
    return np.sqrt(d2.min(1)).mean()

z0 = rng.standard_normal((1500, 2))
for s in [1, 2, 5, 50]:
    print(f'steps={s:2d}  quality (mean NN distance to data) = {nn_dist(sample(net, z0, s), data):.3f}')

With a **random** noise-to-data coupling the marginal field is curved (paths cross), so 1-2 Euler steps are rough and quality keeps improving up to ~50 steps. Straightening the field is what will fix that.

## 3 - Reflow: retrain on the model's own couplings to straighten paths

Reflow replaces the random pairing with the coupling the model *actually* induces (integrate `z -> x_hat` with many steps), then retrains on those straight-line targets. The trajectories straighten, so few-step sampling improves.

In [ ]:
z_rf = rng.standard_normal((40000, 2))
d_rf = sample(net, z_rf, steps=50)                   # model's own endpoints
X2, V2 = cfm_batch(z_rf, d_rf, rng)
net2 = train_field(X2, V2)

print('after reflow:')
for s in [1, 2, 5, 50]:
    print(f'steps={s:2d}  quality = {nn_dist(sample(net2, z0, s), data):.3f}')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for row, (model, name) in enumerate([(net, 'random coupling'), (net2, 'after reflow')]):
    for col, s in enumerate([1, 2, 5, 50]):
        ax = axes[row, col]
        pts = sample(model, z0, s)
        ax.scatter(data[:, 0], data[:, 1], s=3, c='#334155', alpha=0.5)
        ax.scatter(pts[:, 0], pts[:, 1], s=4, c='#14b8a6' if row else '#6366f1', alpha=0.6)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'{name}, {s} step' + ('s' if s > 1 else ''), fontsize=9)
plt.tight_layout(); plt.show()

**What to notice.** Top row (random coupling): 1-2 steps miss the moons badly; only many steps recover them. Bottom row (after reflow): even 1-2 steps land close to the data — the field was straightened, so the coarse solver keeps up. Integration error tracks *curvature*, not distance.

## 4 - Visualize the 'threads' composing the 'rope'

In [ ]:
zz, dd = make_pairs(data, 60, np.random.default_rng(1))
fig, ax = plt.subplots(figsize=(6, 5))
for i in range(len(zz)):
    ax.plot([zz[i, 0], dd[i, 0]], [zz[i, 1], dd[i, 1]], color='#475569', lw=0.6, alpha=0.7)
ax.scatter(zz[:, 0], zz[:, 1], s=14, c='#94a3b8', label='noise z')
ax.scatter(dd[:, 0], dd[:, 1], s=14, c='#14b8a6', label='data d')
ax.set_title('Conditional targets: straight threads whose average is the marginal field')
ax.legend(fontsize=8); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

Each thin line is one trivial conditional target (velocity `d - z`). Individually they are straight; averaged over all pairs they compose the complex marginal flow. **Train on the threads, get the rope.**

## 5 - Your turn

Implement the conditional flow-matching target. Fill in the two `# TODO(you)` lines.

In [ ]:
def my_cfm_target(z, d, t):
    # x_t : the point at progress t on the straight path from z to d
    # v   : the conditional velocity (constant along that path)
    x_t = None  # TODO(you)
    v = None    # TODO(you)
    return x_t, v

_z = np.array([[0.0, 0.0], [1.0, -1.0]])
_d = np.array([[2.0, 2.0], [ -1.0, 3.0]])
_t = np.array([[0.25], [0.75]])
_xt, _v = my_cfm_target(_z, _d, _t)
assert np.allclose(_xt, (1 - _t) * _z + _t * _d)
assert np.allclose(_v, _d - _z)
# at t=0 the path is at the noise point; at t=1 it is at the data point
assert np.allclose(my_cfm_target(_z, _d, np.zeros((2, 1)))[0], _z)
assert np.allclose(my_cfm_target(_z, _d, np.ones((2, 1)))[0], _d)
print('Correct - straight-line path and constant velocity.')

<details>
<summary>Solution</summary>

```python
def my_cfm_target(z, d, t):
    x_t = (1 - t) * z + t * d
    v = d - z
    return x_t, v
```

The velocity has no `t` dependence: on a straight path you move at a constant rate. That is the entire training target.
</details>

## Key takeaways

- **Flow matching learns a velocity field** that transports noise to data along an ODE; the network outputs a *motion*, not a noise estimate.
- **Conditional flow matching** trains on straight-line targets `v = d - z`; averaging over pairs recovers the intractable marginal field.
- **Straight trajectories integrate in few steps** because ODE error tracks curvature, not distance; **reflow** straightens couplings so 1-2 steps suffice.
- Reflow compounds self-training error if overused -- cap the rounds and finish with a realism-scored (perceptual/adversarial) objective.

**Next:** [Diffusion Models](https://ml-viz-ruby.vercel.app/courses/generative-models/05-diffusion-models) for the denoising view, and [Modern Generative AI](https://ml-viz-ruby.vercel.app/courses/generative-models/06-vit-and-modern-genai) for latent-diffusion / SD-family systems built on rectified flow.